In [ ]:
# Setup — imports and paths
import sys
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import librosa

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from src.config import config
from src.data_loader import DataLoader
from src.detectors import Pipeline
from src.evaluation import Evaluator
from src.utils import extract_if_needed, save_versioned_submission

config.paths.base_dir = project_root
config.paths.data_raw_dir = project_root / "data" / "raw"
config.paths.data_processed_dir = project_root / "data" / "processed"
config.paths.submissions_dir = project_root / "submissions"

config.paths.data_raw_dir.mkdir(parents=True, exist_ok=True)
config.paths.data_processed_dir.mkdir(parents=True, exist_ok=True)
config.paths.submissions_dir.mkdir(parents=True, exist_ok=True)

print(f"Project root: {project_root}")
print(f"Setup complete: sr={config.audio.sample_rate}, method={config.onset.method}")

In [ ]:
# Extract datasets
datasets = [
    ("train", config.paths.data_raw_dir / "train.zip", config.paths.data_processed_dir / "train"),
    ("test", config.paths.data_raw_dir / "test.zip", config.paths.data_processed_dir / "test"),
    ("extra_onsets", config.paths.data_raw_dir / "train_extra_onsets.zip", config.paths.data_processed_dir / "train_extra_onsets"),
    ("extra_tempobeats", config.paths.data_raw_dir / "train_extra_tempobeats.zip", config.paths.data_processed_dir / "train_extra_tempobeats"),
]

for name, zip_path, dest in datasets:
    if zip_path.exists():
        extracted = extract_if_needed(zip_path, dest)
        print(f"{name}: {'extracted' if extracted else 'already exists'}")
    else:
        print(f"{name}: ZIP not found")

In [ ]:
# Load all data
loader = DataLoader()

train_dir = config.paths.data_processed_dir / "train"
train_data = loader.load_train(train_dir)
print(f"Training: {len(train_data)} files")

extra_onsets_dir = config.paths.data_processed_dir / "train_extra_onsets"
extra_onsets = loader.load_extra_onsets(extra_onsets_dir) if extra_onsets_dir.exists() else {}
print(f"Extra onsets: {len(extra_onsets)} files")

extra_tempobeats_dir = config.paths.data_processed_dir / "train_extra_tempobeats"
extra_tempobeats = loader.load_extra_tempobeats(extra_tempobeats_dir) if extra_tempobeats_dir.exists() else {}
print(f"Extra tempo/beats: {len(extra_tempobeats)} files")

all_train = {**train_data, **extra_onsets, **extra_tempobeats}
print(f"Total training: {len(all_train)} files")

test_dir = config.paths.data_processed_dir / "test"
test_data = loader.load_test(test_dir)
print(f"Test: {len(test_data)} files")

In [ ]:
# Parameters — set all config values here only
# Onset (EXP-010/011 superflux+whiten; EXP-015 fusion; EXP-020 learned CNN)
config.onset.method = "superflux"
config.onset.multiband = True            # fallback path when fusion is off
config.onset.n_bands = 2
config.onset.merge_tol_ms = 15.0
config.onset.fusion = True               # EXP-015: superflux + complex-domain ODF fusion (fallback if CNN off/absent)
config.onset.fusion_odfs = ("superflux", "complex")
config.onset.threshold = 0.055           # EXP-015 fusion optimum on 277-file set
config.onset.cnn = True                  # EXP-020: PyTorch onset CNN (needs torch + models/onset_cnn.pt; else falls back to fusion)
config.onset.cnn_delta = 0.30            # peak-pick delta on the CNN activation
config.audio.superflux_gamma = 200.0
config.audio.superflux_mu = 3
config.audio.whiten = True
config.audio.whiten_decay = 0.995
config.audio.whiten_floor = 0.10

# Beat (EXP-005/006: tight Gaussian DP; EXP-012: octave select; EXP-018: BLSTM activation)
config.beat.dp_transition_width = 0.10
config.beat.dp_transition_lambda = 1.0
config.beat.beat_octave_select = True
config.beat.beat_octave_gate = 78.0
config.beat.learned = True               # EXP-018: BLSTM beat activation (needs torch + models/beat_blstm.pt; else falls back to log-mel flux)
config.beat.learned_decode = "B"         # comb_fusion tempo + octave-select over the learned activation

# Tempo (EXP-007/008: log-mel AC, search floor + comb x DFT fusion)
config.beat.tempo_search_min = 60.0
config.beat.tempo_method = "comb_fusion"
config.beat.tempo_comb_harmonics = 2

pipeline = Pipeline()
print(f"Onset: cnn={config.onset.cnn} (delta {config.onset.cnn_delta}), fusion={config.onset.fusion} "
      f"odfs={config.onset.fusion_odfs}, threshold={config.onset.threshold}")
print(f"Beat:  octave_select={config.beat.beat_octave_select} (gate={config.beat.beat_octave_gate}), "
      f"learned={config.beat.learned} (decode {config.beat.learned_decode})")
print(f"Tempo: method={config.beat.tempo_method}, harmonics={config.beat.tempo_comb_harmonics}, "
      f"search_min={config.beat.tempo_search_min}")

In [ ]:
# Validate — evaluate on full training set (127 files)
import mir_eval

val_items = train_data  # all 127 annotated files

val_predictions = {}
for stem, info in val_items.items():
    y, sr = loader.load_audio(info['wav'])
    if y is None:
        continue
    onsets, beats, tempos = pipeline.process_file(y, sr)
    val_predictions[stem] = {
        'onsets': onsets.tolist(),
        'beats': beats.tolist(),
        'tempo': tempos
    }

onset_scores, beat_scores, tempo_scores = [], [], []

for stem, gt in val_items.items():
    if stem not in val_predictions:
        continue
    pred = val_predictions[stem]

    if gt.get('onsets') and pred.get('onsets'):
        f, _, _ = mir_eval.onset.f_measure(
            np.array(gt['onsets']), np.array(pred['onsets']), window=0.05
        )
        onset_scores.append(f)

    if gt.get('beats') and pred.get('beats'):
        f = mir_eval.beat.f_measure(np.array(gt['beats']), np.array(pred['beats']))
        beat_scores.append(f)

    if gt.get('tempo') and pred.get('tempo'):
        gt_tempo = gt['tempo']
        if len(gt_tempo) == 1:
            ref_tempi = np.array([gt_tempo[0] / 2, gt_tempo[0]])
            ref_weight = 0.5
        elif len(gt_tempo) == 2:
            ref_tempi = np.array(gt_tempo)
            ref_weight = 0.5
        elif len(gt_tempo) >= 3:
            ref_tempi = np.array(gt_tempo[:2])
            ref_weight = gt_tempo[2]
        else:
            continue
        est_tempi = np.array(pred['tempo'][:2])
        try:
            p_score, _, _ = mir_eval.tempo.detection(ref_tempi, ref_weight, est_tempi, tol=0.08)
            tempo_scores.append(float(p_score))
        except Exception:
            pass

print("VALIDATION RESULTS (127 files)")
print(f"  Onset F1:  {np.mean(onset_scores):.4f}  ({len(onset_scores)} files)")
print(f"  Beat F1:   {np.mean(beat_scores):.4f}  ({len(beat_scores)} files)")
print(f"  Tempo:     {np.mean(tempo_scores):.4f}  ({len(tempo_scores)} files)")
overall = (np.mean(onset_scores) + np.mean(beat_scores) + np.mean(tempo_scores)) / 3
print(f"  MEAN:      {overall:.4f}")

In [ ]:
# Validate — ONSET GENERALIZATION (leaderboard-realistic)
# The 127-file onset score above is over-optimistic: it overfits the small
# annotated train set. The true generalization benchmark is the 277-file
# combined set (127 train + 150 extra-onset), which tracks the leaderboard
# onset score almost exactly (see EXP-014/015). Always read THIS number when
# judging an onset change, not the 127-file one.
extra_onset_f1 = []
for stem, info in extra_onsets.items():
    y, sr = loader.load_audio(info['wav'])
    if y is None:
        continue
    onsets = pipeline.onset_detector.detect(y, sr)  # onset-only: fast, skips beat/tempo
    gt = info.get('onsets')
    if gt and len(onsets):
        f, _, _ = mir_eval.onset.f_measure(np.array(gt), np.array(onsets), window=0.05)
        extra_onset_f1.append(f)

onset_277 = onset_scores + extra_onset_f1
print("ONSET GENERALIZATION")
print(f"  Onset F1 (127 train):    {np.mean(onset_scores):.4f}   (over-optimistic)")
if extra_onset_f1:
    print(f"  Onset F1 (150 extra):    {np.mean(extra_onset_f1):.4f}")
    print(f"  Onset F1 (277 combined): {np.mean(onset_277):.4f}   <- tracks leaderboard")
else:
    print("  (extra_onsets not loaded — run the Data cells to extract train_extra_onsets)")
print("  Beat/tempo generalize at least as well (leaderboard tempo runs higher than val).")

In [ ]:
# Submit — generate versioned submission for test set
submission = {}
for stem, info in test_data.items():
    y, sr = loader.load_audio(info['wav'])
    if y is None:
        submission[stem] = {'onsets': [], 'beats': [], 'tempo': []}
        continue
    onsets, beats, tempos = pipeline.process_file(y, sr)
    submission[stem] = {
        'onsets': onsets.tolist(),
        'beats': beats.tolist(),
        'tempo': tempos
    }

pred_path = save_versioned_submission(
    predictions=submission,
    submissions_dir=config.paths.submissions_dir,
    experiment_id="EXP-020",
    val_scores={"onset_cnn_fair_c127": 0.7881, "beat_blstm_fair": 0.7335, "tempo": 0.7698},
    notes="EXP-020 onset CNN (delta 0.30) + EXP-018 beat BLSTM (decode B) + comb_fusion tempo. Needs torch + models/{onset_cnn,beat_blstm}.pt.",
)
print(f"Processed {len(submission)} test files")
print(f"Upload {pred_path} to https://challenges.cp.jku.at")

In [ ]:
# Visualize results
def visualize(wav_path, onsets, beats, tempos, title=""):
    y, sr = librosa.load(wav_path, sr=config.audio.sample_rate)
    time = np.arange(len(y)) / sr
    
    plt.figure(figsize=(14, 4))
    plt.plot(time, y, alpha=0.6, color='steelblue', linewidth=0.8)
    for o in onsets:
        plt.axvline(x=o, color='red', alpha=0.7, linewidth=0.8, linestyle='--')
    for b in beats:
        plt.axvline(x=b, color='green', alpha=0.7, linewidth=0.8)
    plt.xlabel('Time (s)')
    plt.title(f'{title} | Tempo: {tempos[0]:.1f} BPM')
    plt.xlim(0, min(10, time[-1]))
    plt.tight_layout()
    plt.show()

if val_items:
    stem = list(val_items.keys())[0]
    gt = val_items[stem]
    pred = val_predictions[stem]
    print(f"File: {stem}")
    print(f"GT onsets: {len(gt['onsets'])} | Pred: {len(pred['onsets'])}")
    print(f"GT beats: {len(gt['beats'])} | Pred: {len(pred['beats'])}")
    print(f"GT tempo: {gt['tempo']} | Pred: {pred['tempo']}")
    visualize(gt['wav'], pred['onsets'], pred['beats'], pred['tempo'], stem)

In [ ]:
# Sweep — onset threshold search (restores config on exit)
# Note: EXP-015 multi-ODF fusion (superflux+complex) shifts the F1-optimal
# threshold up to ~0.055 (the complex channel adds peaks, needing a higher
# delta to trim false positives). Sweep grid centred accordingly.
# IMPORTANT: this sweep optimises the 127-file score, which overfits. The
# chosen default (0.055) was tuned on the 277-file ONSET GENERALIZATION cell
# above. Treat the 127-file best below as diagnostic only.
_saved = {
    'threshold': config.onset.threshold, 'method': config.onset.method,
    'gamma': config.audio.superflux_gamma, 'mu': config.audio.superflux_mu,
    'multiband': config.onset.multiband, 'n_bands': config.onset.n_bands,
    'merge_tol': config.onset.merge_tol_ms,
    'fusion': config.onset.fusion, 'fusion_odfs': config.onset.fusion_odfs,
    'whiten': config.audio.whiten, 'whiten_decay': config.audio.whiten_decay,
    'whiten_floor': config.audio.whiten_floor,
    'dp_width': config.beat.dp_transition_width, 'dp_lambda': config.beat.dp_transition_lambda,
    'tempo_search_min': config.beat.tempo_search_min, 'tempo_method': config.beat.tempo_method,
    'comb_h': config.beat.tempo_comb_harmonics,
    'octave_select': config.beat.beat_octave_select, 'octave_gate': config.beat.beat_octave_gate,
}


def run_sweep(thresholds, items=None):
    """Sweep onset threshold. Pass items={**train_data, **extra_onsets} to
    sweep on the 277-file onset set (generalization); default is 127-file
    train (over-optimistic, diagnostic only)."""
    sweep_items = items if items is not None else val_items
    results = []
    for thresh in thresholds:
        config.onset.threshold = thresh
        pipeline_sw = Pipeline()

        onset_f1s, beat_f1s, tempo_sc = [], [], []
        for stem, gt in sweep_items.items():
            y, sr = loader.load_audio(gt['wav'])
            if y is None:
                continue
            has_bt = bool(gt.get('beats')) or bool(gt.get('tempo'))
            if has_bt:
                o, b, t = pipeline_sw.process_file(y, sr)
                pred = {'onsets': o.tolist(), 'beats': b.tolist(), 'tempo': t}
            else:
                # onset-only file (extra_onsets): skip beat/tempo for speed
                pred = {'onsets': pipeline_sw.onset_detector.detect(y, sr).tolist(),
                        'beats': [], 'tempo': []}
            if gt.get('onsets') and pred['onsets']:
                # mir_eval.onset.f_measure returns (f_measure, precision, recall)
                f, _, _ = mir_eval.onset.f_measure(
                    np.array(gt['onsets']), np.array(pred['onsets']), window=0.05
                )
                onset_f1s.append(f)
            if gt.get('beats') and pred['beats']:
                f = mir_eval.beat.f_measure(np.array(gt['beats']), np.array(pred['beats']))
                beat_f1s.append(f)
            if gt.get('tempo') and pred['tempo']:
                gt_tempo = gt['tempo']
                if len(gt_tempo) == 1:
                    ref_tempi = np.array([gt_tempo[0] / 2, gt_tempo[0]])
                    ref_weight = 0.5
                elif len(gt_tempo) == 2:
                    ref_tempi = np.array(gt_tempo)
                    ref_weight = 0.5
                elif len(gt_tempo) >= 3:
                    ref_tempi = np.array(gt_tempo[:2])
                    ref_weight = gt_tempo[2]
                else:
                    continue
                est_tempi = np.array(pred['tempo'][:2])
                try:
                    p_score, _, _ = mir_eval.tempo.detection(ref_tempi, ref_weight, est_tempi, tol=0.08)
                    tempo_sc.append(float(p_score))
                except Exception:
                    pass

        mean_onset = np.mean(onset_f1s) if onset_f1s else 0
        mean_beat = np.mean(beat_f1s) if beat_f1s else 0
        mean_tempo = np.mean(tempo_sc) if tempo_sc else 0
        results.append({'threshold': thresh, 'onset_f1': mean_onset, 'n_onset': len(onset_f1s),
                        'beat_f1': mean_beat, 'tempo': mean_tempo})
        print(f"thresh={thresh:.4f}: Onset={mean_onset:.4f} ({len(onset_f1s)} files)")

    if results:
        best = max(results, key=lambda x: x['onset_f1'])
        print(f"\nBEST onset: threshold={best['threshold']}  Onset F1={best['onset_f1']:.4f}  "
              f"({best['n_onset']} files)")
    return results


# Sweep on the 277-file onset set (generalization benchmark, not 127-only).
sweep_items = {**train_data, **extra_onsets}
thresholds = [0.045, 0.050, 0.055, 0.060, 0.070]
results = run_sweep(thresholds, items=sweep_items)

# Restore config
config.onset.threshold = _saved['threshold']
config.onset.method = _saved['method']
config.audio.superflux_gamma = _saved['gamma']
config.audio.superflux_mu = _saved['mu']
config.onset.multiband = _saved['multiband']
config.onset.n_bands = _saved['n_bands']
config.onset.merge_tol_ms = _saved['merge_tol']
config.onset.fusion = _saved['fusion']
config.onset.fusion_odfs = _saved['fusion_odfs']
config.audio.whiten = _saved['whiten']
config.audio.whiten_decay = _saved['whiten_decay']
config.audio.whiten_floor = _saved['whiten_floor']
config.beat.dp_transition_width = _saved['dp_width']
config.beat.dp_transition_lambda = _saved['dp_lambda']
config.beat.tempo_search_min = _saved['tempo_search_min']
config.beat.tempo_method = _saved['tempo_method']
config.beat.tempo_comb_harmonics = _saved['comb_h']
config.beat.beat_octave_select = _saved['octave_select']
config.beat.beat_octave_gate = _saved['octave_gate']
print(f"\nConfig restored: threshold={config.onset.threshold}, "
      f"fusion={config.onset.fusion}, whiten={config.audio.whiten}, "
      f"tempo_method={config.beat.tempo_method}, octave_select={config.beat.beat_octave_select}")